# 21 — Comparing Multiple Studies

Load the same sensor channel from several bearings at once and overlay their degradation curves on one interactive figure.

**Dataset**: XJTU-SY — all 5 bearings in the 35 Hz / 12 kN condition (Bearing 2_1 … 2_5)  
**API**: `wrapper.compare_studies()` · `plotter.plot_multi_lifecycle()`

In [ ]:
import warnings, logging, sys
from pathlib import Path

warnings.filterwarnings("ignore")
logging.getLogger("isa_phm").setLevel(logging.ERROR)

WRAPPER_ROOT = Path("..").resolve()
if str(WRAPPER_ROOT) not in sys.path:
    sys.path.insert(0, str(WRAPPER_ROOT))

from isa_phm import ISAWrapper
from isa_phm.plotter import ISAPlotter
from bokeh.io import output_notebook
from bokeh.plotting import show as bokeh_show
output_notebook()

ISA_JSON = Path(r"G:\ISA\Datasets\XJTU-SY_Bearing_Datasets\XJTU-SY_Bearing_Datasets\XJTU-SY Bearing Datasets-ISA-PHM-Out.json")
print("Exists:", ISA_JSON.exists())

In [ ]:
wrapper = ISAWrapper(ISA_JSON, strict_validation=False, cache_maxsize=20)
plotter = ISAPlotter()

print(f"{wrapper.investigation_overview().n_studies} studies in dataset")

## 1. Load lifecycle features for all 5 bearings at once

`compare_studies()` selects the same assay from each study and loads all lifecycle features in parallel.  
No manual for-loop needed — the result is a `dict[study_title, DataFrame]` ready for plotting.

Only the studies you pass are included.

In [ ]:
BEARINGS = ["Bearing 2_1", "Bearing 2_2", "Bearing 2_3", "Bearing 2_4", "Bearing 2_5"]

# assay_id=1 → first assay (horizontal accelerometer) of each study
lc_dict = wrapper.compare_studies(
    BEARINGS,
    assay_id=1,
    file_type="raw",
    n_workers=8,
)

print(f"{len(lc_dict)} bearings loaded:")
for name, lc in lc_dict.items():
    print(f"  {name}: {len(lc)} runs")

## 2. RMS lifecycle comparison

Overlay all five RMS curves. Click a legend entry to hide/show individual bearings.

In [ ]:
fig = plotter.plot_multi_lifecycle(
    lc_dict,
    feature="rms",
    title="35 Hz / 12 kN — RMS Degradation (Bearing 2_1 … 2_5)",
)
bokeh_show(fig)

## 3. Kurtosis comparison

Kurtosis shows the timing of the impulsive fault onset — which bearing failed earliest?

In [ ]:
fig = plotter.plot_multi_lifecycle(
    lc_dict,
    feature="kurtosis",
    title="35 Hz / 12 kN — Kurtosis (Bearing 2_1 … 2_5)",
)
bokeh_show(fig)

## 4. Crest factor comparison

Crest factor often rises *before* RMS, making it useful for early detection.

In [ ]:
fig = plotter.plot_multi_lifecycle(
    lc_dict,
    feature="crest_factor",
    title="35 Hz / 12 kN — Crest Factor (Bearing 2_1 … 2_5)",
)
bokeh_show(fig)

## 5. Cross-condition comparison

`study.compare_with()` lets you start from one study and add others — including bearings from different operating conditions.

The base study (`self`) is always included.

In [ ]:
study_11 = wrapper.study("Bearing 1_1")   # 35 Hz / 12 kN
study_31 = wrapper.study("Bearing 3_1")   # 40 Hz / 10 kN — longest trajectory (2538 runs)

cross_dict = study_11.compare_with(
    [study_31],
    assay_id=1,
    file_type="raw",
    n_workers=8,
)

fig = plotter.plot_multi_lifecycle(
    cross_dict,
    feature="rms",
    title="Cross-condition RMS Degradation",
)
bokeh_show(fig)